# Azure Supplement — Memory & Compute Budgets

> **Source notes:** `memory-budgets.md`  
> **Azure services:** Azure ML compute clusters, GPU VM SKUs (NC-series, ND-series, NV-series)

This supplement demonstrates VRAM estimation and memory profiling on Azure GPU instances.  
It answers: **"Which Azure GPU SKU do I need for my model?"**

**What this supplement covers:**

| Step | Description | Azure Service |
|------|-------------|---------------|
| 1 | Azure credentials setup (empty strings — fill before running) | Azure Identity |
| 2 | Azure SDK setup | `azure-ai-ml`, `azure-identity` |
| 3 | Azure GPU SKU comparison (VRAM, cost, performance) | Azure Pricing API |
| 4 | VRAM estimation for Llama-3-8B on Azure | — |
| 5 | Memory profiling on Azure ML compute cluster | Azure ML |
| 6 | Batch size vs. VRAM tradeoff on Azure | — |
| 7 | Choosing the right Azure GPU for inference vs. training | — |

**Prerequisites:**
- Azure subscription with quota for GPU VMs (NC, ND, or NV series)
- Azure ML workspace created (can use free tier for profiling, but GPU quota required for actual runs)
- Azure CLI installed and authenticated: `az login`

## Step 0 — Azure Credentials Setup

**⚠️ IMPORTANT: Fill in your Azure credentials before running.**

These cells are left empty to pass Git hooks. Replace `""` with your actual values:

```python
AZURE_SUBSCRIPTION_ID = "your-subscription-id-here"
AZURE_RESOURCE_GROUP = "your-resource-group"
AZURE_ML_WORKSPACE = "your-ml-workspace-name"
AZURE_REGION = "eastus"  # or your preferred region
```

**Tip:** Use Azure Key Vault for production. For learning, you can use environment variables:
```bash
export AZURE_SUBSCRIPTION_ID="..."
export AZURE_RESOURCE_GROUP="..."
```

In [ ]:
# TODO: Implement this cell


## Step 1 — Azure SDK Setup

Install Azure ML SDK and verify authentication.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Step 2 — Azure GPU SKU Comparison

Compare VRAM, cost, and availability of Azure GPU VM SKUs.

**Azure GPU VM families:**
- **NC-series** — NVIDIA Tesla K80, M60 (older, budget GPUs)
- **NCv3-series** — NVIDIA Tesla V100 (16 GB or 32 GB VRAM)
- **ND-series** — NVIDIA Tesla P40 (24 GB VRAM, training-optimized)
- **NDv2-series** — NVIDIA Tesla V100 (32 GB VRAM × 8 GPUs, multi-GPU training)
- **NC A100 v4-series** — NVIDIA A100 (40 GB or 80 GB VRAM, latest generation)

**Key decision:** Match VRAM requirements from Ch.2 formulas to Azure SKU.

In [ ]:
# TODO: Implement this cell


## Step 3 — VRAM Estimation for Llama-3-8B on Azure

Use the Ch.2 VRAM formulas to estimate which Azure GPU SKU fits your model.

**Recall from Ch.2:**
```
VRAM_inference = Params + KV Cache + Activations
VRAM_training = Params + Optimizer States + Gradients + Activations
```

We'll calculate for Llama-3-8B at different precisions and batch sizes.

In [ ]:
# TODO: Implement this cell


## Step 4 — Training VRAM Estimation (Adam Optimizer)

For training, add optimizer states (Adam = 2× params in FP32) and gradients (1× params).

**Formula from Ch.2:**
```
VRAM_training = Params (FP16) + Optimizer States (FP32) + Gradients (FP16) + Activations
                = 2P + 8P + 2P + 8 GB = 12P + 8 GB (for FP16 params, FP32 optimizer)
```

In [ ]:
# TODO: Implement this cell


## Step 5 — Memory Profiling on Azure ML Compute Cluster (Dry Run)

Demonstrate how to profile actual VRAM usage on an Azure ML compute cluster.

**Note:** This is a **local simulation** since spinning up Azure GPU VMs for every learner is expensive.  
In production, you would:
1. Create an Azure ML compute cluster with the target GPU SKU
2. Submit a profiling job that loads your model and logs VRAM usage
3. Use `torch.cuda.memory_allocated()` to track actual memory consumption

**For learning purposes**, we'll simulate the profiling output here.

In [ ]:
# TODO: Implement this cell


## Step 6 — Batch Size vs. VRAM Tradeoff (Azure Cost Optimization)

Visualize how batch size affects VRAM and throughput on Azure.

**Key insight from Ch.2:** Throughput scales with batch size, but so does VRAM consumption (via KV cache).  
Larger Azure GPU SKUs enable higher batch sizes → better throughput → lower cost per request.

In [ ]:
# TODO: Implement this cell


## Step 7 — Choosing the Right Azure GPU (Decision Tree)

**Decision tree for Azure GPU selection:**

```
┌─────────────────────────────────────────────────────────────┐
│ What are you doing?                                         │
└─────────────────────────────────────────────────────────────┘
    │
    ├─ Inference (Llama-3-8B, FP16, batch=1)
    │  → Standard_NC6s_v3 (V100 16GB, $3.06/hr)
    │  → Fits: 22 GB < 24 GB ✅, but batch=1 only
    │
    ├─ Inference (Llama-3-8B, FP16, batch=4 for throughput)
    │  → Standard_NC24ads_A100_v4 (A100 40GB, $3.67/hr)
    │  → Fits: 28 GB < 40 GB ✅, batch=4 → 4× throughput
    │
    ├─ Inference (Llama-3-8B, INT4, batch=8)
    │  → Standard_ND6s (P40 24GB, $2.07/hr)
    │  → Fits: 14 GB < 24 GB ✅, INT4 saves $1/hr vs V100
    │
    ├─ Training (Llama-3-8B, LoRA fine-tuning)
    │  → Standard_NC6s_v3 (V100 16GB, $3.06/hr)
    │  → LoRA adapters = 2 GB → total 18 GB ✅
    │
    ├─ Training (Llama-3-8B, full fine-tuning, Adam)
    │  → Standard_NC96ads_A100_v4 (A100 80GB, $10.75/hr)
    │  → Full training = 104 GB → need A100 80GB ✅
    │
    └─ Inference (Llama-2-70B, INT4)
       → Standard_NC96ads_A100_v4 (A100 80GB, $10.75/hr)
       → 70B INT4 = 35 GB params + 21 GB KV = 56 GB ✅
```

In [ ]:
# TODO: Implement this cell


## Step 8 — Production Checklist: Deploying on Azure ML

**Before deploying your model to Azure ML compute:**

1. ✅ **Calculate exact VRAM requirements** (use formulas from this notebook)
2. ✅ **Choose Azure GPU SKU** with 10-20% headroom (not 100% utilization)
3. ✅ **Request GPU quota** in Azure subscription (NC/ND series often have low default quotas)
4. ✅ **Test with smaller batch size first** (batch=1) before scaling up
5. ✅ **Monitor GPU memory** during inference using `torch.cuda.memory_allocated()`
6. ✅ **Set up auto-scaling** in Azure ML to handle variable load
7. ✅ **Use INT8/INT4 quantization** (Ch.3) to reduce costs by 2-4×

**Common mistakes:**
- ❌ Forgetting KV cache scaling (batch × seq_len)
- ❌ Running at 95%+ VRAM utilization (causes fragmentation OOM)
- ❌ Not checking Azure GPU quota before deploying
- ❌ Choosing cheapest SKU without headroom (false economy — OOM kills throughput)

**Next steps:**
- Ch.3: Quantization (reduce VRAM by 4× with INT4)
- Ch.4: Parallelism (split across multiple Azure GPUs with DeepSpeed)
- Ch.5: Inference Optimization (deploy vLLM on Azure Kubernetes Service)

## Summary — What You Learned

You now know how to:

1. ✅ **Map VRAM requirements to Azure GPU SKUs** (NC, ND, NC A100 series)
2. ✅ **Estimate inference and training memory** for any model size and precision
3. ✅ **Optimize batch size** to balance throughput and cost on Azure
4. ✅ **Choose the cheapest Azure GPU** that fits your model (with headroom)
5. ✅ **Avoid OOM errors** by calculating exact VRAM before deploying

**Key formulas from Ch.2 (applied to Azure):**

| Metric | Formula |
|--------|---------|
| Inference VRAM | `Params + KV Cache + Activations` |
| Training VRAM | `Params + Optimizer States (2× FP32) + Gradients + Activations` |
| KV Cache | `2 × layers × heads × head_dim × seq_len × batch_size × bytes` |
| Azure cost/month | `sku_hourly_rate × 24 × 30` |

**Critical insight:** A larger Azure GPU (A100 vs V100) costs 20% more per hour, but enables 4× higher batch size → 3× lower cost per request.

**Bridge to Ch.3:** Quantization reduces Azure costs by 4× (INT4) while maintaining quality. The same VRAM formulas apply — just change `bytes_per_param` from 2.0 to 0.5.